# SPK-6 — Adaptive Query Execution (AQE) deep-dive

**Break → Detect → Fix → Prove.** AQE re-plans a query *at runtime* using the real statistics each
shuffle produces, instead of trusting the optimizer's compile-time guesses. We run **three demos**,
each **AQE-off vs AQE-on**, reading the effect from `df.explain()` and a before/after metrics table.

This is the deep-dive companion to the skew flagship [`SPK-1`](spk1_data_skew.ipynb): SPK-1 fixes skew
by hand and uses AQE skew-join as *one* of three remedies — here we open AQE up and watch **all** of
what it does for free (coalesce, skew-split, SMJ→broadcast re-optimize), plus the two places it costs you.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch the **SQL / DataFrame** tab as the
cells run.

**Laptop-safe:** data is *generated lazily* (10–20M rows) and only `count()`-ed — never collected or
written — so nothing fills memory or disk. AQE behavior is about task/partition counts and plan
shape, not memory, so the default **tuned** box is fine (no need for `make up-constrained`). Nothing
to delete at the end.

See the [Spark-UI guide](../docs/spark-ui-guide.md), and the [troubleshooting sheet](../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import skewed_keys, key_dimension, uniform_keys
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — AQE shows up in the SQL / DataFrame tab's final plan.
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — A tiny helper to read AQE off the plan

We stay **Connect-safe**: DataFrame/SQL, `spark.conf`, and `df.explain()` only — never
`spark.sparkContext` / RDDs. The AQE tells live in the *final physical plan*:

- **`AdaptiveSparkPlan isFinalPlan=true`** — AQE ran and settled on a final plan.
- **`AQEShuffleRead`** — a coalesced (fewer partitions) or skew-split (more partitions) shuffle read.
- the **join operator** (`SortMergeJoin` vs `BroadcastHashJoin`) — what AQE chose at runtime.

`explain()` only shows `isFinalPlan=true` *after* the query has executed, so we call it on a frame
we've already `count()`-ed.

In [ ]:
def scan_plan(df, note=""):
    """Print df's physical plan and flag the AQE markers (Connect-safe: explain() only)."""
    plan = df._explain_string(mode="formatted") if hasattr(df, "_explain_string") else None
    if plan is None:
        # Fallback that works on all Spark Connect builds: capture explain() output.
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            df.explain(mode="formatted")
        plan = buf.getvalue()
    print(f"--- physical plan {note} ---")
    print(plan)
    print("AQE markers:",
          "isFinalPlan=true"     if "isFinalPlan=true" in plan else "(pre-AQE plan; over Connect explain() is non-final — confirm via metrics + SQL UI)",
          "| AQEShuffleRead"      if "AQEShuffleRead"   in plan else "| AQE split not shown in pre-AQE plan (check skew ratio in compare() + SQL UI)",
          "| BroadcastHashJoin"   if "BroadcastHashJoin" in plan else
          "| SortMergeJoin"       if "SortMergeJoin"     in plan else "")
    return plan

## What each demo shows — AQE off vs on at a glance

| Demo | AQE **off** (broken) — plan / metrics | AQE **on** (fixed) — plan / metrics |
|------|---------------------------------------|-------------------------------------|
| **(a) Coalesce** | `Exchange` feeds **~200 tasks**; Stages tab shows hundreds of millisecond tasks | plan shows **`AQEShuffleRead coalesced`**; **task count drops to single digits** |
| **(b) Skew-join** | `SortMergeJoin`; Tasks Summary: **Duration Max >> Median** on the reduce stage | plan shows **`AQEShuffleRead ... skewed`** — hot partition split; max-vs-median flattens |
| **(c) Re-optimize** | `SortMergeJoin` with **two `Exchange` nodes** | plan flips to **`BroadcastHashJoin`** — the big side's `Exchange` disappears |

Confirm AQE fired by reading **`AdaptiveSparkPlan isFinalPlan=true`** + **`AQEShuffleRead`** in the plan text; the SQL/DataFrame tab surfaces the same.

## Demo (a) — Coalesce shuffle partitions

A heavily-*filtered* fact aggregated with `spark.sql.shuffle.partitions = 200`. The shuffle produces
**200 mostly-empty partitions → 200 tiny tasks** for a few rows of result. AQE's
`coalescePartitions` collapses them to a handful sized to the *real* post-filter data.

**Break it (AQE off):** the `constrained` profile sets `adaptive.enabled=false` and
`coalescePartitions.enabled=false`. We pin 200 partitions and aggregate a tiny slice of a 20M-row fact.

In [ ]:
N_ROWS = 20_000_000

# A uniform fact; filter to a tiny slice, then aggregate -> a small post-shuffle result.
fact_a = uniform_keys(spark, n_rows=N_ROWS, n_keys=2_000, key_col="customer_id")
agg_a  = (fact_a.filter(F.col("customer_id") < 20)        # throw away ~99% of rows
                .groupBy("customer_id")
                .agg(F.sum("amount").alias("total")))

apply_profile(spark, "constrained", **{"spark.sql.shuffle.partitions": "200"})  # 200 tiny tasks
m_a_off = measure(spark, "coalesce: AQE off (200)", lambda: agg_a.count())
scan_plan(agg_a, "(a) AQE OFF")
print("\nmetrics:", m_a_off)

**Detect** (Spark UI → SQL/DataFrame): AQE-off the `Exchange` feeds ~200 tasks (Stages tab shows
hundreds of millisecond tasks). **Fix it (AQE on):** `tuned` turns on `coalescePartitions`. The plan
gains an **`AQEShuffleRead coalesced`** node and the task count drops to single digits — even though
we *still* request 200 partitions, AQE right-sizes them at runtime.

In [ ]:
apply_profile(spark, "tuned", **{"spark.sql.shuffle.partitions": "200"})  # AQE will coalesce these
m_a_on = measure(spark, "coalesce: AQE on", lambda: agg_a.count())
scan_plan(agg_a, "(a) AQE ON")
print("\nmetrics:", m_a_on)
print("\n>>> Prove (a): task count should fall sharply with AQE on")
compare([m_a_off, m_a_on])

## Demo (b) — Skew-join split *(cross-ref [`SPK-1`](spk1_data_skew.ipynb))*

The exact SPK-1 pathology: a 90%-hot-key fact sort-merge-joined onto its dimension. Every hot-key row
lands in one reduce partition → **one straggler task** (`Duration` Max ≫ Median). AQE's skew-join
**splits** that fat partition at runtime — the *automatic* version of SPK-1's manual salting.

**Break it (AQE off):** `constrained` (AQE off, broadcast off) forces the sort-merge join.

> **Why aggregate `amount` instead of a bare `.count()`?** See [SPK-1](spk1_data_skew.ipynb) — the reasoning (AQE skew-detection needs bytes in the hot partition, so we carry a real payload rather than pruning to key-only) is spelled out there; it applies identically here.

In [ ]:
N_COLD = 2_000
fact_b = skewed_keys(spark, n_rows=N_ROWS, hot_key_fraction=0.90,
                     n_cold_keys=N_COLD, hot_key=0, key_col="customer_id")
dim_b  = key_dimension(spark, n_keys=N_COLD + 1, key_col="customer_id")
join_b = fact_b.join(dim_b, on="customer_id", how="inner")

apply_profile(spark, "constrained")          # AQE off, broadcast off -> sort-merge join, one straggler
m_b_off = measure(spark, "skew: AQE off (SMJ)", lambda: join_b.agg(F.sum("amount")).collect())
scan_plan(join_b, "(b) AQE OFF")
print(f"\nSKEW RATIO (max/median): {m_b_off['skew_ratio']}x   <- one straggler doing ~all the work")

**Detect** (Spark UI → SQL/DataFrame → reduce Stage → Tasks → Summary Metrics): **Max ≫ Median**
task time, one fat **Shuffle Read** task. **Fix it (AQE on):** turn on skew-join and — because our
data is tiny — **lower the skew threshold to 16 MB** (the 256 MB default never trips on a laptop; see
[`SPK-1` §7](spk1_data_skew.ipynb)). Broadcast stays **off** so it remains a sort-merge join; the plan
gains an **`AQEShuffleRead ... skewed`** node and the skew ratio collapses.

In [ ]:
apply_profile(spark, "tuned", **{
    "spark.sql.autoBroadcastJoinThreshold": "-1",                                  # keep it a sort-merge join
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "1m",          # laptop scale (SPK-1 trick)
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "2",
})
m_b_on = measure(spark, "skew: AQE skew-join", lambda: join_b.agg(F.sum("amount")).collect())
scan_plan(join_b, "(b) AQE ON")
print("\n>>> Prove (b): skew ratio should collapse from tens-of-x toward ~1-3x")
compare([m_b_off, m_b_on])

## Demo (c) — Runtime re-optimization: SMJ → broadcast

A big fact joined to a *small*, broadcastable dimension — but with broadcast **disabled** at plan
time, so Catalyst commits to a **sort-merge join** that shuffles *both* sides. Once the build side
materializes and AQE sees it's tiny, AQE **re-optimizes the join to a broadcast** — the big side's
`Exchange` vanishes.

**Break it (AQE off):** `constrained` (AQE off, broadcast off) → `SortMergeJoin` with two `Exchange` nodes.

In [ ]:
fact_c = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_COLD, key_col="customer_id")
dim_c  = key_dimension(spark, n_keys=N_COLD, key_col="customer_id")   # small -> broadcastable
join_c = fact_c.join(dim_c, on="customer_id", how="inner")

apply_profile(spark, "constrained")          # AQE off, broadcast off -> SMJ, both sides shuffled
m_c_off = measure(spark, "reopt: AQE off (SMJ)", lambda: join_c.count())
scan_plan(join_c, "(c) AQE OFF")

# Fix it (AQE on): broadcast re-enabled (default 10 MB). AQE sees the small side at runtime and
# flips SortMergeJoin -> BroadcastHashJoin; the big side's Exchange disappears (shuffle ~ 0).
apply_profile(spark, "tuned")
m_c_on = measure(spark, "reopt: AQE on (broadcast)", lambda: join_c.count())
scan_plan(join_c, "(c) AQE ON")
print("\n>>> Prove (c): shuffle bytes should drop to ~0 as SMJ -> broadcast")
compare([m_c_off, m_c_on])

## When AQE costs you — overhead & non-determinism

AQE is rarely *wrong*, but it isn't free:

1. **Planning overhead on tiny queries.** Re-planning after every shuffle is negligible on a big job
 but is pure overhead on a trivial one. Below we time a tiny aggregation **AQE off vs on** — on a
 query this small, AQE-on can be *no faster or slightly slower*. (Spark skips AQE entirely only for
 queries with **no** `Exchange`/subquery.)
2. **Run-to-run non-determinism.** The **coalesced partition count depends on the runtime data
 size**, so it can vary run-to-run and across environments — a trap if a downstream step, test, or
 file-count assertion expects a fixed number of partitions (ties to the streaming small-files
 concern in `STR-3`).

In [ ]:
# A tiny query where AQE's re-planning is overhead, not benefit.
tiny = (uniform_keys(spark, n_rows=200_000, n_keys=50, key_col="k")
        .groupBy("k").agg(F.sum("amount").alias("total")))

apply_profile(spark, "constrained")                       # AQE off
m_tiny_off = measure(spark, "tiny: AQE off", lambda: tiny.count())

apply_profile(spark, "tuned")                             # AQE on (re-plans after the shuffle)
m_tiny_on = measure(spark, "tiny: AQE on", lambda: tiny.count())

print(">>> On a trivial query, AQE-on is no faster (and the partition count can vary run-to-run):")
compare([m_tiny_off, m_tiny_on])

## Takeaways & "in real production…"

- **AQE is on by default in Spark 3.2+ / 4.x.** Three fixes for free: **coalesce** (too many tiny
 partitions), **skew-join split** (one straggler key), and **SMJ → broadcast re-optimization** (a
 mis-sized join strategy).
- **Read it in the plan:** AQE-on plans say **`AdaptiveSparkPlan isFinalPlan=true`** and carry
 **`AQEShuffleRead`** nodes (annotated `coalesced` / `skewed`); the displayed plan is the *final*
 one. The **SQL / DataFrame** tab confirms the join operator and coalesced partition count
 (see [`docs/spark-ui-guide.md`](../docs/spark-ui-guide.md)).
- **Where to still intervene:** AQE adds a little planning overhead on trivial queries, and its
 coalesced partition counts are **non-deterministic** — don't hard-code an expected partition/file
 count downstream. On small data, lower `skewJoin.skewedPartitionThresholdInBytes` (the SPK-1 caveat).
- **In production:** keep AQE enabled; set `spark.sql.shuffle.partitions` sanely and let coalesce trim
 it; lean on AQE skew-join before hand-salting (`SPK-1`); set `autoBroadcastJoinThreshold`
 deliberately so the SMJ→broadcast re-optimization can fire.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We just restore
the production-tuned safety nets.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")